# Занятие 4. Классификация недвижимости: Домашнее задание

Сначала разберём, как из признаков получается вероятность и класс, на одном ручном примере. Затем обучим готовую логистическую регрессию и сравним её с моделью, которая всегда отвечает самым частым классом. На каждом шаге будем разбирать, что делает код и зачем это нужно.

## Правила выполнения

- Запустите подготовку, затем выполняйте задания по порядку. Раскомментируйте заготовки и замените `...` своим кодом.
- Используйте исходные строки и порядок столбцов CSV: не сортируйте, не удаляйте строки и не сбрасывайте индекс.
- Цель — `above_market`: `1` означает «выше рыночного ориентира», `0` — «на уровне или ниже».
- Одно разбиение для всех моделей: `test_size=0.2`, `random_state=42`, `stratify=y`.
- Обучайте преобразования и модели только на train. На test применяйте уже обученные преобразования и получайте прогнозы.
- Как в лекции, используйте `LogisticRegression(C=np.inf, max_iter=2000, random_state=42)`; остальные параметры стандартные. Для справки: `C=np.inf` отключает регуляризацию.
- Сначала обучите модели на train, затем сравните их на одном test. Не подбирайте параметры модели и порог по результатам test.
- Положительный класс для всех метрик это `1`. В вызовах `precision_score`, `recall_score` и `f1_score` указывайте `zero_division=0`: если формула метрики даёт деление на ноль, библиотека вернёт 0 без предупреждения. В задании 8 разберём, почему это происходит у постоянной модели.
- Промежуточные значения не округляйте. В числовых ответах используйте точку, без единиц измерения и знака `%`.

## Подготовка

Нужны `numpy`, `pandas`, `matplotlib`, `plotly`, `scikit-learn` и, только для скачивания, `kagglehub`.
В IDE выберите ядро своего окружения; устанавливайте библиотеки в это же ядро.

Используем **data.csv** из [Dag Real Estate Classification на Kaggle](https://www.kaggle.com/datasets/guleibatabdullaeva/dag-real-estate-classification).

По умолчанию код скачает датасет через `kagglehub`: структура ваших папок не имеет значения.
Если CSV уже скачан, укажите путь к нему в `DATA_PATH` в ячейке подготовки. Можно использовать абсолютный путь; относительный путь отсчитывается от текущей рабочей папки, которую код выводит перед загрузкой.
Схема проверяется: CSV из занятия 3 с ценами для этой работы не подходит.

Сохраняйте имена переменных из шаблонов: последняя ячейка проверяет каждое задание отдельно и выводит подсказки. После исправления перезапустите зависимые ячейки и проверку.


In [ ]:
# При необходимости установите библиотеки в выбранное ядро:
# %pip install -q numpy pandas matplotlib plotly scikit-learn kagglehub

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

SEED = 42
# Оставьте None для автоматического скачивания или укажите свой путь к CSV.
# Например: DATA_PATH = Path.home() / "Downloads" / "data.csv"
# В Windows можно указать Path(r"C:\Users\student\Downloads\data.csv").
DATA_PATH = None
print("Текущая рабочая папка:", Path.cwd())
if DATA_PATH is None:
    import kagglehub

    downloaded = Path(kagglehub.dataset_download(
        "guleibatabdullaeva/dag-real-estate-classification"
    ))
    files = list(downloaded.rglob("data.csv"))
    assert len(files) == 1, f"Ожидался один data.csv; найдены: {list(downloaded.rglob('*.csv'))}"
    DATA_PATH = files[0]

else:
    DATA_PATH = Path(DATA_PATH).expanduser()
    if not DATA_PATH.is_file():
        raise FileNotFoundError(
            f"CSV не найден: {DATA_PATH.resolve()}. Укажите свой путь в DATA_PATH "
            "или оставьте None для скачивания."
        )

print("Загружаем CSV:", DATA_PATH)
df = pd.read_csv(DATA_PATH)
numeric_features = [
    "Всего этажей в доме", "Общая площадь", "Этаж",
    "rooms", "passenger_lift_count",
]
assert {"above_market", *numeric_features}.issubset(df.columns), "Нужен CSV для классификации."
assert not {"price", "price_per_m2_calc"}.intersection(df.columns), "Ценовые признаки создают утечку."
assert df.index.is_unique and df.isna().sum().sum() == 0
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns)
assert set(df["above_market"].unique()) == {0, 1}
# Снимок нужен только для проверки, что исходные данные не изменены.
_source_df = df.copy(deep=True)
print("Размер таблицы:", df.shape)
display(df.head())


## Задание 1. Баланс классов

Сначала познакомимся с ответами в данных: сколько квартир относится к каждому классу? Если одного класса заметно больше, можно часто угадывать ответ, просто выбирая его каждый раз. Поэтому перед обучением посмотрим на баланс классов — позже он поможет понять результат самой простой модели.

По всем строкам `df` посчитайте количество квартир каждого класса (`class_counts`, Series с индексами 0, 1), долю класса 1 (`positive_share`) и самый частый класс (`majority_class`).

**Вопрос:** сколько процентов квартир относится к классу 1? Сохраните это число в `positive_percent`. **Формат:** три знака после точки.

In [ ]:
# class_counts = ...  # value_counts(), затем sort_index()
# positive_share = ...
# majority_class = ...
# positive_percent = ...
# Выведите positive_percent с тремя знаками после точки.


## Задание 2. Признаки, цель и одно разбиение

Теперь отделим то, что модель получает на вход, от того, что она должна предсказать. В `X` оставим характеристики квартиры, а в `y` — правильный ответ `above_market`. Если оставить ответ среди признаков, модель сможет просто подсмотреть его.

Разделим квартиры на две части: на train модель будет учиться, а на test мы проверим её на незнакомых строках. `stratify=y` сохраняет примерно одинаковые доли классов в обеих частях. Обе модели получат одно и то же разбиение, чтобы сравнение было честным.

Создайте `X`, исключив только `above_market`, и `y` с этой целью. Одним вызовом `train_test_split` получите `X_train`, `X_test`, `y_train`, `y_test` с параметрами `test_size=0.2, random_state=SEED, stratify=y`. Проверьте совпадение индексов X и y и отсутствие пересечений train/test.

**Вопрос:** сколько квартир в test? Сохраните в `test_size`. **Формат:** целое число.

In [ ]:
# X = ...
# y = ...
# X_train, X_test, y_train, y_test = ...
# Добавьте assert для индексов и отсутствия пересечений.
# test_size = ...


## Задание 3. StandardScaler и MinMaxScaler без утечки

У признаков разные масштабы: площадь может измеряться десятками и сотнями, а число комнат — единицами. Сейчас попробуем два способа привести числовые признаки к сопоставимому масштабу. `StandardScaler` вычитает среднее и делит на стандартное отклонение; `MinMaxScaler` переводит обучающие значения в диапазон от 0 до 1.

Вызов `fit` запоминает нужные статистики, а `transform` использует их для пересчёта значений. Статистики берём только из train: для test применяем уже готовое преобразование. Здесь сравним оба способа отдельно, а в нашу модель дальше включим `StandardScaler`.

Для пяти столбцов `numeric_features` обучите на `X_train` два отдельных объекта: `standard = StandardScaler()` и `minmax = MinMaxScaler()`. Получите массивы `train_standard`, `test_standard` и `test_minmax`. Для test вызывайте только `transform`.

Вычислите среднее площади на train (`area_train_mean`) и масштабированную StandardScaler площадь первой тестовой квартиры (`first_area_standard`). У StandardScaler используется стандартное отклонение с `ddof=0`; Min-Max на новых данных может выходить за 0–1.

**Вопрос:** чему равна `first_area_standard`? **Формат:** три знака после точки.

In [ ]:
# 1. Создайте два независимых преобразователя.
# standard = StandardScaler()
# minmax = MinMaxScaler()
# 2. Выберите только пять количественных столбцов.
# train_numeric = X_train[numeric_features]
# test_numeric = X_test[...]  # тот же список столбцов и тот же порядок
# 3. fit изучает статистики только train; transform применяет их к данным.
# standard.fit(...)
# minmax.fit(...)
# train_standard = standard.transform(...)
# test_standard = standard.transform(...)
# test_minmax = minmax.transform(...)
# 4. Среднее берите по исходной площади train, не по масштабированным данным.
# area_train_mean = X_train["Общая площадь"]. ...  # метод среднего
# area_position = numeric_features.index(...)  # название столбца площади
# first_area_standard = test_standard[..., ...]  # строка 0, столбец area_position
# print(f"{first_area_standard:.3f}")


## Задание 4. От линейной суммы к вероятности и потере

Сейчас напишем своими руками, как логистическая регрессия получает прогноз для одной квартиры. Сначала она умножает признаки на веса и складывает результат со свободным членом — получается число `z`. Затем сигмоида превращает это число в значение от 0 до 1, которое модель использует как вероятность класса 1. По порогу 0.5 из вероятности получаем класс.

Чтобы увидеть каждый шаг, возьмём только один признак — площадь — и зададим веса сами. Например, вероятность 0.8 при нашем пороге даст прогноз «класс 1», но правильность прогноза проверяется по настоящему ответу. Затем посчитаем потерю: она показывает, насколько прогноз вероятности согласуется с ответом; уверенная ошибка даёт большую потерю.

Здесь мы только выполняем расчёт с заданными весами. В задании 6 библиотека сама подберёт веса на train уже по всем признакам. В задании 7 вызовем `predict_proba` и получим вероятности без ручного вычисления сигмоиды.

Возьмите `x = first_area_standard` из задания 3. Для учебных весов вычислите `z_demo = -1 + 2 * x`, затем сигмоиду `p_demo = 1 / (1 + exp(-z_demo))`.

Получите `class_demo` при пороге 0.5: класс 1, если `p_demo > 0.5`, иначе 0 (как `predict` при равенстве в этой бинарной модели). Для настоящего ответа первой тестовой квартиры вычислите вручную `loss_demo`:

$$\ell(y,p)=-[y\ln p+(1-y)\ln(1-p)].$$

Это заданные учебные веса, обучение здесь не требуется.

**Вопрос:** чему равна `loss_demo`? **Формат:** три знака после точки.

In [ ]:
# z_demo = ...
# p_demo = ...
# class_demo = ...
# true_demo = ...  # первая строка y_test через iloc
# loss_demo = ...  # np.log — натуральный логарифм


## Задание 5. Постоянный классификатор

Прежде чем обучать нашу модель, зададим простой ориентир: что получится, если вообще не смотреть на характеристики квартиры и каждый раз отвечать самым частым классом?

Именно так работает `DummyClassifier(strategy="most_frequent")`: на train он узнаёт, какой класс встречается чаще, и затем всегда выбирает его. Площадь, этаж и другие признаки на его ответ не влияют. Позже сравним его с логистической регрессией на одних и тех же тестовых квартирах: помогает ли нашей модели использование признаков?

Сейчас найдём только долю правильных ответов этой простой модели на train. Финальное сравнение обеих моделей проведём на test в задании 8.

Обучите `baseline = DummyClassifier(strategy="most_frequent")` только на train. Получите его постоянный класс `baseline_class` через прогноз для `X_train.iloc[:1]` и долю этого класса в `y_train` — `baseline_train_accuracy`. До задания 7 прогнозы на test не нужны.

**Вопрос:** чему равна `baseline_train_accuracy`? **Формат:** три знака после точки. Это accuracy на train, а не финальная оценка.

In [ ]:
# baseline = DummyClassifier(strategy=...)  # most_frequent: самый частый класс
# baseline.fit(..., ...)  # признаки train и ответы train
# Прогноз для одной строки — массив длины 1; возьмите элемент с индексом 0.
# baseline_class = baseline.predict(X_train.iloc[:1])[...]
# Сравнение даёт булеву Series: её среднее равно доле совпадений.
# baseline_train_accuracy = (y_train == ...).mean()
# print(f"{baseline_train_accuracy:.3f}")


## Задание 6. Масштабирование и модель в Pipeline

Мы уже разобрали расчёт вероятности вручную. Теперь соберём готовую модель из `scikit-learn`: `LogisticRegression` сама подберёт веса по обучающим примерам. При `fit` она будет искать веса, с которыми вероятности лучше согласуются с правильными ответами train.

Объединим подготовку данных и классификатор в `Pipeline` — последовательность шагов. Сначала `ColumnTransformer` масштабирует пять числовых столбцов и пропускает остальные без изменений, затем классификатор учится на результате. Так при будущих прогнозах к новым квартирам автоматически применяется то же преобразование.

В `model.fit` передаём исходные признаки train: Pipeline сам обучит scaler и модель. Массивы из задания 3 были нужны, чтобы отдельно разобраться с масштабированием. В конце заглянем внутрь Pipeline и проверим, какое среднее площади запомнил его scaler.

Создайте `preprocessor = ColumnTransformer` с преобразованием `("numeric", StandardScaler(), numeric_features)` и `remainder="passthrough"`. Так остальные признаки сохранятся без масштабирования.

Создайте `model = Pipeline` с шагами `("preparation", preprocessor)` и `("classifier", LogisticRegression(C=np.inf, max_iter=2000, random_state=SEED))`. Обучите Pipeline на исходном `X_train`, `y_train`. Отдельно масштабированные массивы из задания 3 сюда не передавайте.

Найдите среднее площади внутри обученного Pipeline: `pipeline_area_mean` из `model.named_steps["preparation"].named_transformers_["numeric"].mean_`.

**Вопрос:** чему равно `pipeline_area_mean`? **Формат:** три знака после точки.

In [ ]:
# preprocessor = ColumnTransformer(
#     transformers=[("numeric", StandardScaler(), ...)],  # пять числовых столбцов
#     remainder=...,  # passthrough сохраняет остальные столбцы
# )
# model = Pipeline([
#     ("preparation", ...),  # созданный выше preprocessor
#     ("classifier", LogisticRegression(C=..., max_iter=2000, random_state=SEED)),
# ])
# model.fit(..., ...)  # исходные X_train и y_train, без ручного масштабирования
# Доступ к обученному scaler: сначала шаг Pipeline, затем преобразователь.
# pipeline_scaler = model.named_steps["preparation"].named_transformers_["numeric"]
# area_position = numeric_features.index(...)  # название столбца площади
# pipeline_area_mean = pipeline_scaler.mean_[...]
# print(f"{pipeline_area_mean:.3f}")


## Задание 7. Финальные прогнозы: класс и вероятность

Модель обучена. Теперь дадим ей тестовые квартиры, ответы на которые она не использовала при обучении. Для каждой квартиры получим два результата: `predict_proba` вернёт вероятности классов, а `predict` — выбранный класс. Это готовый библиотечный вариант тех шагов, которые мы выполняли вручную в задании 4.

Нам нужна вероятность именно класса 1, поэтому найдём его позицию в `model.classes_`. Рядом положим настоящий ответ и отметим несовпадения. Самая простая модель тоже сделает прогнозы на этих же квартирах — дальше сравним их качество.

Модель и её параметры зафиксированы. Получите `pred = model.predict(X_test)`, `probability` — вероятности именно класса 1 через `predict_proba` и позицию класса в `model.classes_`, а также `pred_baseline` от постоянного классификатора.

Создайте таблицу `comparison` с индексами `y_test` и столбцами `Истина`, `P(1)`, `Прогноз`, `Ошибка` (булев признак несовпадения). Покажите первые десять строк. Сохраните `first_probability` для первой тестовой квартиры.

**Вопрос:** чему равна `first_probability`? **Формат:** три знака после точки.

In [ ]:
# pred = model.predict(...)  # признаки test
# Порядок колонок predict_proba соответствует model.classes_.
# positive_column = list(model.classes_).index(...)  # нужен класс 1
# probability = model.predict_proba(...)[:, ...]  # все строки, positive_column
# pred_baseline = baseline.predict(...)
# comparison = pd.DataFrame({
#     "Истина": y_test,
#     "P(1)": ...,
#     "Прогноз": ...,
#     "Ошибка": y_test != ...,  # сравнение истины и pred
# }, index=...)  # сохраните исходные индексы y_test
# first_probability = probability[...]  # первая позиция в массиве, индекс 0
# display(comparison.head(10).round(3))
# print(f"{first_probability:.3f}")


## Задание 8. Матрица ошибок и четыре метрики

Прогнозы готовы — теперь проверим, чего стоят наши модели. Одной доли правильных ответов мало: модель, которая всегда выбирает частый класс, может выглядеть неплохо по accuracy и при этом не находить ни одной квартиры класса 1.

Матрица ошибок покажет, какие именно ответы перепутаны: FP — обычную квартиру назвали «выше ориентира», FN — пропустили квартиру, которая действительно выше ориентира. Precision отвечает на вопрос «как часто мы правы, когда предсказываем класс 1?», recall — «какую долю настоящих единиц мы нашли?». F1 объединяет precision и recall в одно число.

Посчитаем одинаковые метрики для обеих моделей на одном test. Так станет видно, что изменилось по сравнению с постоянным ответом и научилась ли логистическая регрессия находить класс 1.

Получите `cm = confusion_matrix(y_test, pred, labels=[0, 1])` и извлеките `tn, fp, fn, tp = cm.ravel()`. Строки матрицы — истина, столбцы — прогноз.

Создайте `metrics_table`: индекс `Постоянный класс`, `Логистическая регрессия`; столбцы `Accuracy`, `Precision`, `Recall`, `F1`. Все метрики считайте на одном `y_test`, для класса 1. Постройте матрицу ошибок через `ConfusionMatrixDisplay`.

**Зачем нужен `zero_division=0`?** Постоянный классификатор в этой работе всем предсказывает класс 0. Поэтому у него нет ни верных, ни ошибочных прогнозов класса 1: `TP = 0`, `FP = 0`. Формула precision даёт `TP / (TP + FP) = 0 / 0` — такое деление не определено. Параметр `zero_division=0` говорит библиотеке: «в этом случае запиши в таблицу 0 и не выводи предупреждение».

Например: `precision_score(y_test, pred_baseline, zero_division=0)`. Указывайте этот параметр также в `recall_score` и `f1_score`: у этих метрик тоже бывают случаи с нулевым знаменателем. Он применяется только в таких случаях и не меняет обычный расчёт. У `accuracy_score` такого параметра нет.

Сохраните `model_f1` из таблицы. **Вопрос:** чему равен F1 логистической регрессии? **Формат:** три знака после точки.

In [ ]:
# cm = ...
# tn, fp, fn, tp = ...
# metrics_table = ...
# model_f1 = ...
# Покажите таблицу и матрицу ошибок.


## Демонстрация. Как порог вероятности меняет ошибки

Мы видели, что модель сначала выдаёт вероятность, а затем превращает её в класс по порогу. Теперь оставим вероятности прежними и будем двигать только порог. Так можно увидеть, почему при одних настройках модель чаще поднимает ложную тревогу, а при других — чаще пропускает класс 1. Переобучать модель для этого не нужно.

Запустите готовую ячейку и перетаскивайте ползунок **«Порог»**. **Ничего дописывать не нужно.**

Слева показаны шесть искусственных объектов: положение точки — вероятность класса 1, строка — настоящий класс. Справа от пунктирной линии прогноз равен 1, на линии и слева — 0: правило `p > порог`. Подписи точек показывают исход классификации, а столбцы справа — число ошибок.

- **FP** — ложная тревога: настоящий класс 0, прогноз 1.
- **FN** — пропуск: настоящий класс 1, прогноз 0.
- **TN / TP** — верно предсказанные классы 0 / 1.

Сравните пороги 0.30, 0.50 и 0.80: с ростом порога ложных тревог становится меньше, а пропусков — больше. Вероятности и веса модели при этом не меняются. Это учебный пример, без подбора порога по test; рабочий порог выбирают внутри train и фиксируют до финальной оценки.


In [ ]:
# Готовая интерактивная демонстрация: выполнять задание не требуется.
# Plotly включает ползунок прямо в график; сервер и ipywidgets не нужны.
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML

y_example = np.array([0, 0, 1, 0, 1, 1])
p_example = np.array([0.10, 0.35, 0.40, 0.55, 0.70, 0.90])
object_labels = np.array([f"№{i}" for i in range(1, 7)])


def threshold_state(threshold):
    example_pred = (p_example > threshold).astype(int)
    outcomes = np.where(
        y_example == 1,
        np.where(example_pred == 1, "TP", "FN"),
        np.where(example_pred == 1, "FP", "TN"),
    )
    colors = {"TN": "#2563eb", "TP": "#15803d", "FP": "#d97706", "FN": "#dc2626"}
    counts = [int(np.sum(outcomes == name)) for name in ["FP", "FN"]]
    return {
        "labels": [f"{label}: {outcome}" for label, outcome in zip(object_labels, outcomes)],
        "colors": [colors[outcome] for outcome in outcomes],
        "counts": counts,
    }


initial = threshold_state(0.5)
threshold_figure = make_subplots(
    rows=1, cols=2, column_widths=[0.7, 0.3], horizontal_spacing=0.12,
    subplot_titles=("Вероятности и настоящие классы", "Количество ошибок"),
)
threshold_figure.add_trace(go.Scatter(
    x=p_example, y=y_example, mode="markers+text",
    text=initial["labels"], textposition="top center",
    marker=dict(size=17, color=initial["colors"], line=dict(width=1, color="#333333")),
    customdata=object_labels,
    hovertemplate="%{customdata}<br>P(1) = %{x:.2f}<br>Истина = %{y}<br>%{text}<extra></extra>",
    showlegend=False,
), row=1, col=1)
threshold_figure.add_trace(go.Bar(
    x=["FP: ложные тревоги", "FN: пропуски"], y=initial["counts"],
    marker_color=["#d97706", "#dc2626"], text=initial["counts"],
    textposition="outside", showlegend=False,
    hovertemplate="%{x}: %{y}<extra></extra>",
), row=1, col=2)

# update меняет подписи и цвета точек, высоты столбцов и положение линии.
steps = []
for threshold in np.round(np.linspace(0, 1, 101), 2):
    state = threshold_state(threshold)
    steps.append(dict(
        method="update", label=f"{threshold:.2f}",
        args=[{
            "text": [state["labels"], state["counts"]],
            "marker.color": [state["colors"], ["#d97706", "#dc2626"]],
            "y": [y_example.tolist(), state["counts"]],
        }, {"shapes[0].x0": float(threshold), "shapes[0].x1": float(threshold)}],
    ))
threshold_figure.update_layout(
    title="Двигайте порог: как меняются FP и FN",
    template="plotly_white", height=470,
    margin=dict(l=65, r=35, t=85, b=120),
    shapes=[dict(
        type="line", xref="x", yref="y", x0=0.5, x1=0.5,
        y0=-0.3, y1=1.3, line=dict(color="#555555", width=2, dash="dash"),
    )],
    sliders=[dict(
        active=50, currentvalue=dict(prefix="Порог: "),
        steps=steps, ticklen=0, pad=dict(t=60),
    )],
)
threshold_figure.update_xaxes(title_text="Вероятность класса 1: P(1)", range=[-0.06, 1.06], row=1, col=1)
threshold_figure.update_yaxes(
    title_text="Настоящий класс", tickvals=[0, 1],
    ticktext=["0: на уровне / ниже", "1: выше"], range=[-0.4, 1.5], row=1, col=1,
)
threshold_figure.update_yaxes(title_text="Число объектов", range=[0, 3.8], dtick=1, row=1, col=2)
# Библиотека встроена: ползунок работает без интернета и после сохранения вывода.
display(HTML(threshold_figure.to_html(
    full_html=False, include_plotlyjs=True, config={"displayModeBar": False, "responsive": True},
)))


## Проверка ДЗ

Запустите после заданий 1–8. Проверяются исходные данные, разбиение, статистики scaler, структура Pipeline и численные результаты. Для каждого задания выводится отдельный результат. Демонстрация с ползунком не проверяется. Проверка не оценивает внешний вид графиков; сравните их с лекцией самостоятельно. После исправления выполните все зависимые ячейки. Численные сравнения допускают небольшую погрешность вычислений.

In [ ]:
# Каждое задание проверяется отдельно. Проверка не обучает модели заново.
# close запрещает неявное сравнение массивов разной формы и пропускает только конечные значения.
def close(actual, expected, atol=1e-7):
    actual, expected = np.asarray(actual), np.asarray(expected)
    return (actual.shape == expected.shape and np.isfinite(actual).all()
            and np.isfinite(expected).all()
            and np.allclose(actual, expected, atol=atol, rtol=1e-7))

errors = []

# Задание 1
try:
    assert df.equals(_source_df), "Исходная таблица была изменена."
    assert class_counts.equals(df["above_market"].value_counts().sort_index())
    assert close(positive_share, df["above_market"].mean())
    assert majority_class == class_counts.idxmax()
    assert close(positive_percent, 100 * positive_share)
    assert np.array_equal(class_counts, [3125, 2366]), "Проверьте версию исходного CSV."
    print("Задание 1: верно")
except Exception as error:
    errors.append(1)
    print("Задание 1: ошибка. Считайте по всем строкам; доля и процент отличаются в 100 раз.")
    print(f"  {type(error).__name__}: {error}")

# Задание 2
try:
    assert X.equals(df.drop(columns=["above_market"])) and y.equals(df["above_market"])
    ref_train, ref_test = train_test_split(df.index, test_size=0.2, random_state=42, stratify=df["above_market"])
    assert X_train.equals(X.loc[ref_train]) and X_test.equals(X.loc[ref_test])
    assert y_train.equals(y.loc[ref_train]) and y_test.equals(y.loc[ref_test])
    assert X_train.index.intersection(X_test.index).empty
    assert test_size == len(X_test)
    assert test_size == 1099
    print("Задание 2: верно")
except Exception as error:
    errors.append(2)
    print("Задание 2: ошибка. Исключите цель и используйте совместное стратифицированное разбиение 80/20 с seed=42.")
    print(f"  {type(error).__name__}: {error}")

# Задание 3
try:
    assert isinstance(standard, StandardScaler) and isinstance(minmax, MinMaxScaler)
    assert standard.n_samples_seen_ == len(X_train) and minmax.n_samples_seen_ == len(X_train)
    assert list(standard.feature_names_in_) == numeric_features
    assert list(minmax.feature_names_in_) == numeric_features
    assert close(standard.mean_, X_train[numeric_features].mean())
    assert close(standard.var_, X_train[numeric_features].var(ddof=0))
    assert close(minmax.data_min_, X_train[numeric_features].min())
    assert close(minmax.data_max_, X_train[numeric_features].max())
    assert close(train_standard, standard.transform(X_train[numeric_features]))
    assert close(test_standard, standard.transform(X_test[numeric_features]))
    assert close(test_minmax, minmax.transform(X_test[numeric_features]))
    assert close(area_train_mean, X_train["Общая площадь"].mean())
    assert close(first_area_standard, (X_test.iloc[0]["Общая площадь"] - area_train_mean) / X_train["Общая площадь"].std(ddof=0))
    print("Задание 3: верно")
except Exception as error:
    errors.append(3)
    print("Задание 3: ошибка. Оба fit выполняются на train; площадь берите в первой строке test, а не по индексу 0.")
    print(f"  {type(error).__name__}: {error}")

# Задание 4
try:
    assert close(z_demo, -1 + 2 * first_area_standard)
    assert close(p_demo, 1 / (1 + np.exp(-z_demo)))
    assert class_demo == int(p_demo > 0.5) and true_demo == y_test.iloc[0]
    assert close(loss_demo, -(true_demo * np.log(p_demo) + (1 - true_demo) * np.log(1 - p_demo)))
    print("Задание 4: верно")
except Exception as error:
    errors.append(4)
    print("Задание 4: ошибка. Сигмоида применяется к линейной сумме, а log loss использует правильный класс и вероятность.")
    print(f"  {type(error).__name__}: {error}")

# Задание 5
try:
    assert isinstance(baseline, DummyClassifier) and baseline.strategy == "most_frequent"
    assert close(baseline.class_prior_, y_train.value_counts(normalize=True).reindex(baseline.classes_))
    assert baseline_class == y_train.value_counts().idxmax()
    assert np.all(baseline.predict(X_train) == baseline_class)
    assert close(baseline_train_accuracy, (y_train == baseline_class).mean())
    print("Задание 5: верно")
except Exception as error:
    errors.append(5)
    print("Задание 5: ошибка. Самый частый класс и его долю определяйте только по y_train.")
    print(f"  {type(error).__name__}: {error}")

# Задание 6
try:
    assert isinstance(model, Pipeline) and list(model.named_steps) == ["preparation", "classifier"]
    prep = model.named_steps["preparation"]
    clf = model.named_steps["classifier"]
    assert isinstance(prep, ColumnTransformer) and prep.remainder == "passthrough"
    assert prep.transformers[0][0] == "numeric" and prep.transformers[0][2] == numeric_features
    assert isinstance(prep.named_transformers_["numeric"], StandardScaler)
    assert isinstance(clf, LogisticRegression) and np.isinf(clf.C)
    assert clf.max_iter == 2000 and clf.random_state == 42 and clf.solver == "lbfgs"
    assert clf.fit_intercept and clf.class_weight is None
    assert list(model.feature_names_in_) == list(X_train.columns)
    assert close(prep.named_transformers_["numeric"].mean_, X_train[numeric_features].mean())
    assert prep.named_transformers_["numeric"].n_samples_seen_ == len(X_train)
    assert close(pipeline_area_mean, area_train_mean)
    assert np.all(clf.n_iter_ < clf.max_iter), "Модель не сошлась: проверьте подготовку."
    print("Задание 6: верно")
except Exception as error:
    errors.append(6)
    print("Задание 6: ошибка. Сохраните имена шагов и параметры из условия; fit должен обучить scaler и классификатор на train.")
    print(f"  {type(error).__name__}: {error}")

# Задание 7
try:
    assert positive_column == list(model.classes_).index(1)
    assert np.shape(probability) == (len(y_test),) and np.isfinite(probability).all()
    assert np.all((probability >= 0) & (probability <= 1))
    assert close(probability, model.predict_proba(X_test)[:, positive_column])
    assert np.array_equal(pred, model.predict(X_test))
    assert np.array_equal(pred, (probability > 0.5).astype(int))
    assert np.array_equal(pred_baseline, baseline.predict(X_test)) and np.all(pred_baseline == baseline_class)
    assert comparison.index.equals(y_test.index)
    assert list(comparison.columns) == ["Истина", "P(1)", "Прогноз", "Ошибка"]
    assert np.array_equal(comparison["Истина"], y_test) and close(comparison["P(1)"], probability)
    assert np.array_equal(comparison["Прогноз"], pred) and np.array_equal(comparison["Ошибка"], y_test != pred)
    assert close(first_probability, probability[0])
    assert np.isclose(first_probability, 0.035, atol=0.005, rtol=0), "Проверьте модель, данные для fit и порядок test."
    print("Задание 7: верно")
except Exception as error:
    errors.append(7)
    print("Задание 7: ошибка. Берите вероятность класса 1 и сохраняйте исходные индексы тестовых строк.")
    print(f"  {type(error).__name__}: {error}")

# Задание 8
try:
    assert np.array_equal(cm, confusion_matrix(y_test, pred, labels=[0, 1]))
    assert np.array_equal([tn, fp, fn, tp], cm.ravel()) and cm.sum() == len(y_test)
    assert list(metrics_table.index) == ["Постоянный класс", "Логистическая регрессия"]
    assert list(metrics_table.columns) == ["Accuracy", "Precision", "Recall", "F1"]
    for name, labels in [("Постоянный класс", pred_baseline), ("Логистическая регрессия", pred)]:
        assert close(metrics_table.loc[name], [accuracy_score(y_test, labels), precision_score(y_test, labels, zero_division=0), recall_score(y_test, labels, zero_division=0), f1_score(y_test, labels, zero_division=0)])
    assert close(model_f1, metrics_table.loc["Логистическая регрессия", "F1"])
    assert np.isclose(model_f1, 0.868, atol=0.005, rtol=0)
    print("Задание 8: верно")
except Exception as error:
    errors.append(8)
    print("Задание 8: ошибка. Проверьте порядок TN, FP, FN, TP и считайте precision/recall/F1 для класса 1.")
    print(f"  {type(error).__name__}: {error}")

if errors:
    print("Исправьте задания:", ", ".join(map(str, errors)))
else:
    print("(=^･ω･^=)👍 Все 8 заданий выполнены верно!")
